In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("diabetes.csv")

In [3]:
df.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.corr()['Outcome']

Pregnancies                 0.221898
Glucose                     0.466581
BloodPressure               0.065068
SkinThickness               0.074752
Insulin                     0.130548
BMI                         0.292695
DiabetesPedigreeFunction    0.173844
Age                         0.238356
Outcome                     1.000000
Name: Outcome, dtype: float64

In [5]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [6]:
X.shape

(768, 8)

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [8]:
X = scaler.fit_transform(X)

In [9]:
X.shape

(768, 8)

In [10]:
from sklearn.model_selection import train_test_split
X_train , X_test,y_train ,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [45]:
import tensorflow 
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Dropout

In [12]:
model = Sequential()
model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

/opt/anaconda3/envs/tensorflow_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:

model.fit(X_train,y_train,batch_size=32,epochs=10,validation_data=(X_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5896 - loss: 0.6809 - val_accuracy: 0.6558 - val_loss: 0.6527
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6792 - loss: 0.6301 - val_accuracy: 0.7143 - val_loss: 0.6064
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7003 - loss: 0.5929 - val_accuracy: 0.7468 - val_loss: 0.5721
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7280 - loss: 0.5657 - val_accuracy: 0.7662 - val_loss: 0.5481
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7378 - loss: 0.5459 - val_accuracy: 0.7727 - val_loss: 0.5294
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7443 - loss: 0.5293 - val_accuracy: 0.7662 - val_loss: 0.5164
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7378 - loss: 0.5169 - val_accuracy: 0.7662 - val_loss: 0.5067
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7443 - loss: 0.5063 - val_accuracy: 0.7792 - val_loss:

In [14]:
# 1. how to select appropriate optimizer
# 2. Number of nodes in a layer
# 3. how to select number of layers
# 4. All in all one model 



In [15]:
import keras_tuner as kt

In [16]:
def build_model(hp):

    model = Sequential()

    model.add(Dense(32,activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))

    optimizer = hp.Choice('optimizer',values =['adam','sgd','rmsprop','adadelta'])

    model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])

    return model



In [17]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [18]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [19]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [20]:
model = tuner.get_best_models(num_models=1)[0]

/opt/anaconda3/envs/tensorflow_env/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [21]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.fit(X_train,y_train,batch_size=32,epochs=100,validation_data=(X_test,y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7655 - loss: 0.4982 - val_accuracy: 0.7922 - val_loss: 0.4891
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7638 - loss: 0.4872 - val_accuracy: 0.7987 - val_loss: 0.4834
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7704 - loss: 0.4808 - val_accuracy: 0.8052 - val_loss: 0.4783
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7671 - loss: 0.4753 - val_accuracy: 0.8052 - val_loss: 0.4751
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7671 - loss: 0.4717 - val_accuracy: 0.7987 - val_loss: 0.4726
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7687 - loss: 0.4682 - val_accuracy: 0.8052 - val_loss: 0.4712
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7655 - loss: 0.4655 - val_accuracy: 0.8117 - val_loss: 0.4687
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7687 - loss: 0.4631 - val_accuracy: 0.8117 - v

In [23]:
def build_model2(hp):
    
    model = Sequential()

    units = hp.Int('units',min_value=8,max_value=128,step=8)

    model.add(Dense(units=units,activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))
    
    model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])

    return model


In [27]:
tuner2 = kt.RandomSearch(build_model2,
                        objective='val_accuracy',
                        max_trials=5,
                        directory='mydir',
                        project_name='Salman Khan')

/opt/anaconda3/envs/tensorflow_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
tuner2.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 01s]
val_accuracy: 0.798701286315918

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 00m 05s


In [29]:
tuner2.get_best_hyperparameters()[0].values

{'units': 128}

In [30]:
model = tuner2.get_best_models(num_models=1)[0]

/opt/anaconda3/envs/tensorflow_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/opt/anaconda3/envs/tensorflow_env/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [31]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6)

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7622 - loss: 0.5269  
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 839us/step - accuracy: 0.7638 - loss: 0.4941
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 869us/step - accuracy: 0.7704 - loss: 0.4790
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 826us/step - accuracy: 0.7899 - loss: 0.4687
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 747us/step - accuracy: 0.7899 - loss: 0.4620
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 668us/step - accuracy: 0.7834 - loss: 0.4573
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 671us/step - accuracy: 0.7785 - loss: 0.4540
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 701us/step - accuracy: 0.7866 - loss: 0.4518
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 687us/step - accuracy: 0.7899 - loss: 0.4499
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 714us/step - accuracy: 0.7883 - loss: 0.4480
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 664us/step - accuracy: 0.7850 - loss: 0.4462
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━

In [32]:
def build_model3(hp):

    model = Sequential()

    model.add(Dense(72,activation='relu',input_dim=8))

    for i in range(hp.Int('num_layers',min_value=1,max_value=10)):

        model.add(Dense(72,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='rmsprop',loss='binary_crossentropy',metrics=['accuracy'])

    return model

In [33]:
tuner3 = kt.RandomSearch(build_model3,
                        objective='val_accuracy',
                        max_trials=3,
                        directory= 'mydir',
                        project_name = 'num_layers'
                        
)

/opt/anaconda3/envs/tensorflow_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [35]:
tuner3.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 01s]
val_accuracy: 0.7727272510528564

Best val_accuracy So Far: 0.8051947951316833
Total elapsed time: 00h 00m 05s


In [46]:
#All in One 
def build_model4(hp):
    model = Sequential()

    counter =0


    for i in range(hp.Int('num_layers',min_value=1,max_value=10)):

        if counter == 0:
            model.add(
                Dense
                (hp.Int('units' + str(i) , min_value=8 , max_value=128),
                activation=hp.Choice('activation' + str(i) , values=['relu','tanh','sigmoid']),
                input_dim=8
                )
            )
            model.add(Dropout(hp.Choice('dropout' + str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        else:

            model.add(
                Dense
                (hp.Int('units' + str(i) , min_value=8 , max_value=128),
                activation=hp.Choice('activation' + str(i) , values=['relu','tanh','sigmoid'])
              
                )
            )
            model.add(Dropout(hp.Choice('dropout' + str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        counter+=1

    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer=hp.Choice('optimizer',values=['rmsprop','adam','sgd','nadam','adadelta']),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

In [47]:
tuner4 = kt.RandomSearch(build_model4,
                         objective='val_accuracy',
                         max_trials=3,
                         directory='mydir',
                         project_name='final1')

/opt/anaconda3/envs/tensorflow_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [48]:
tuner4.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 01s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.6428571343421936
Total elapsed time: 00h 00m 05s


In [49]:
tuner4.get_best_hyperparameters()[0].values

{'num_layers': 2,
 'units0': 71,
 'activation0': 'sigmoid',
 'dropout0': 0.9,
 'optimizer': 'adam',
 'units1': 8,
 'activation1': 'relu',
 'dropout1': 0.1}

In [50]:
model = tuner4.get_best_models(num_models=1)[0]

/opt/anaconda3/envs/tensorflow_env/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [51]:
model.fit(X_train,y_train,epochs=200,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4837 - loss: 0.9318 - val_accuracy: 0.6429 - val_loss: 0.6464
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5293 - loss: 0.8698 - val_accuracy: 0.6429 - val_loss: 0.6552
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5391 - loss: 0.8636 - val_accuracy: 0.6429 - val_loss: 0.6576
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6059 - loss: 0.7546 - val_accuracy: 0.6429 - val_loss: 0.6523
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5765 - loss: 0.7893 - val_accuracy: 0.6429 - val_loss: 0.6382
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6173 - loss: 0.7087 - val_accuracy: 0.6429 - val_loss: 0.6334
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5879 - loss: 0.7309 - val_accuracy: 0.6429 - val_loss: 0.6289
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5961 - loss: 0.7214 - val_accuracy: 0.642